# Case 00 · Tensors & Neural Network Basics

**Goal:** Build the vocabulary you need to co-pilot with an AI agent on any neural network task.

**Runs on:** CPU, under a minute. No GPU needed.

| Section | What you learn |
|---------|---------------|
| 1 | Tensors: create, inspect, reshape, unsqueeze |
| 2 | nn.Linear and nn.Sequential: what layers do |
| 3 | The training loop: forward → loss → backward → step |
| 4 | Putting it together: train a model on sin(x) |
| 5 | The co-pilot cheat sheet |

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}")

---
## 1 · Tensors: the container for everything

A tensor is a multi-dimensional array of numbers. Your data, your model's weights, the gradients, the loss — all tensors.

In [ ]:
# Scalar (0D) — a single number
scalar = torch.tensor(3.14)
print(f"scalar:  value={scalar.item():.2f}  shape={scalar.shape}  ndim={scalar.ndim}")

# Vector (1D) — a list of numbers
vector = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"vector:  shape={vector.shape}  ndim={vector.ndim}")

# Matrix (2D) — a grid
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(f"matrix:  shape={matrix.shape}  ndim={matrix.ndim}")

# 3D tensor — a stack of grids
tensor3d = torch.randn(2, 3, 4)  # random, shape (2, 3, 4)
print(f"3D:      shape={tensor3d.shape}  ndim={tensor3d.ndim}  numel={tensor3d.numel()}")

### unsqueeze and squeeze — adding/removing dimensions

This is **the most common operation** you'll see in deep learning code, and the most confusing at first.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"Original:      shape = {x.shape}")        # (5,)

# unsqueeze(0): add a dimension at position 0 → (1, 5)
# Think: "1 batch of 5 features" — a row
x0 = x.unsqueeze(0)
print(f"unsqueeze(0):  shape = {x0.shape}")        # (1, 5)

# unsqueeze(1): add a dimension at position 1 → (5, 1)
# Think: "5 samples, each with 1 feature" — a column
x1 = x.unsqueeze(1)
print(f"unsqueeze(1):  shape = {x1.shape}")        # (5, 1)

# squeeze: remove all size-1 dimensions
x_back = x1.squeeze()
print(f"squeeze():     shape = {x_back.shape}")    # (5,)

print("\n--- Why this matters ---")
print("nn.Linear(1, 64) expects input shape (batch, 1)")
print(f"x has shape {x.shape} — nn.Linear would fail!")
print(f"x.unsqueeze(1) has shape {x1.shape} — now it works.")

In [ ]:
# reshape / view — rearrange without copying
a = torch.arange(12)  # [0, 1, 2, ..., 11], shape (12,)
print(f"Original: {a.shape}")

b = a.reshape(3, 4)   # 3 rows, 4 cols
print(f"reshape(3,4): {b.shape}")
print(b)

c = a.reshape(2, 2, 3) # 2 x 2 x 3
print(f"\nreshape(2,2,3): {c.shape}")
print(c)

# -1 means "figure it out"
d = a.reshape(-1, 3)  # ? x 3 → 4 x 3
print(f"\nreshape(-1,3): {d.shape}")

---
## 2 · Layers: what nn.Linear and nn.Sequential do

A layer transforms a tensor. `nn.Linear(in, out)` does: `y = Wx + b`

In [ ]:
# One linear layer: 1 feature in, 4 features out
layer = nn.Linear(1, 4)

print(f"Weight shape: {layer.weight.shape}")  # (4, 1)
print(f"Bias shape:   {layer.bias.shape}")    # (4,)
print(f"Total params: {sum(p.numel() for p in layer.parameters())}")

# Feed one sample through
x = torch.tensor([[2.0]])  # shape (1, 1) = 1 sample, 1 feature
y = layer(x)
print(f"\nInput:  {x.shape} → {x.tolist()}")
print(f"Output: {y.shape} → {[round(v, 3) for v in y.tolist()[0]]}")

# Verify: y = x * W^T + b
manual = x @ layer.weight.T + layer.bias
print(f"Manual: {[round(v, 3) for v in manual.tolist()[0]]}")
print("Match!", torch.allclose(y, manual))

In [ ]:
# nn.Sequential: stack layers in order
model = nn.Sequential(
    nn.Linear(1, 64),   # 1 feature → 64 hidden
    nn.Tanh(),           # squish to [-1, 1]
    nn.Linear(64, 64),   # 64 → 64
    nn.Tanh(),
    nn.Linear(64, 1),    # 64 → 1 output
)

print("Model:")
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print("(That's the weight matrices + bias vectors of all 3 Linear layers)")

# Trace the shape through each layer
x = torch.randn(8, 1)  # 8 samples, 1 feature each
print(f"\nInput shape: {x.shape}")
h = x
for i, layer in enumerate(model):
    h = layer(h)
    print(f"  After layer {i} ({layer.__class__.__name__:8s}): {h.shape}")

### Why do we need activation functions?

Without Tanh (or ReLU, GELU, etc.), stacking linear layers is pointless — it's still just one big linear operation.

In [ ]:
x = torch.linspace(-3, 3, 200)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, fn) in zip(axes, [
    ('Tanh', torch.tanh),
    ('ReLU', torch.relu),
    ('Sigmoid', torch.sigmoid),
]):
    ax.plot(x.numpy(), fn(x).numpy(), linewidth=2)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(name)
    ax.set_ylim(-1.5, 1.5)
    ax.grid(alpha=0.2)
plt.suptitle('Common activation functions — the non-linearity that makes deep learning work')
plt.tight_layout()
plt.show()

---
## 3 · The training loop: forward → loss → backward → step

Every training loop in deep learning — including fine-tuning, LoRA, QLoRA — follows the same 4 steps.

In [ ]:
# Generate simple data: y = sin(x)
x = torch.linspace(-math.pi, math.pi, 256).unsqueeze(1)  # (256, 1)
y = torch.sin(x)                                          # (256, 1)

print(f"x shape: {x.shape}  (256 samples, 1 feature each)")
print(f"y shape: {y.shape}  (256 targets, 1 value each)")

# Build a small model
model = nn.Sequential(
    nn.Linear(1, 64), nn.Tanh(),
    nn.Linear(64, 64), nn.Tanh(),
    nn.Linear(64, 1),
)

# Set up the optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()  # Mean Squared Error

print(f"\nOptimizer: Adam (lr=0.001)")
print(f"Loss function: MSE (mean squared error)")
print(f"Trainable params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# === THE 4 STEPS, one at a time, so you see what each does ===

# Step 0: zero old gradients (otherwise they accumulate!)
optimizer.zero_grad()
print("Step 0: zeroed gradients")

# Step 1: FORWARD — run input through model to get prediction
pred = model(x)
print(f"Step 1 (Forward):  pred shape = {pred.shape}, first 3 values = {pred[:3, 0].tolist()}")

# Step 2: LOSS — how wrong is the prediction?
loss = loss_fn(pred, y)
print(f"Step 2 (Loss):     MSE = {loss.item():.6f}")

# Step 3: BACKWARD — compute gradients (how to change each weight to reduce loss)
loss.backward()
first_weight = model[0].weight
print(f"Step 3 (Backward): gradient of first weight = {first_weight.grad[0, 0].item():.6f}")
print(f"  (This tells Adam which direction to nudge this weight)")

# Step 4: UPDATE — optimizer adjusts all weights
old_val = first_weight.data[0, 0].item()
optimizer.step()
new_val = first_weight.data[0, 0].item()
print(f"Step 4 (Update):   weight moved from {old_val:.6f} to {new_val:.6f}")
print(f"  (Change: {new_val - old_val:.6f} — a tiny nudge in the gradient direction)")

---
## 4 · Putting it together: train a model on sin(x)

Now we run the loop many times and watch the loss curve.

In [ ]:
# Fresh model
model = nn.Sequential(
    nn.Linear(1, 64), nn.Tanh(),
    nn.Linear(64, 64), nn.Tanh(),
    nn.Linear(64, 1),
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training data
x = torch.linspace(-math.pi, math.pi, 256).unsqueeze(1)
y = torch.sin(x)

# Train!
losses = []
for step in range(500):
    optimizer.zero_grad()
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 100 == 0 or step == 499:
        print(f"  step {step:3d}  loss = {loss.item():.6f}")

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(losses, color='#ff8a5c')
ax1.set_xlabel('Training step')
ax1.set_ylabel('Loss (MSE)')
ax1.set_title('Loss curve — should go down!')
ax1.set_yscale('log')
ax1.grid(alpha=0.2)

# Prediction vs truth
with torch.no_grad():
    pred = model(x)
ax2.plot(x.numpy(), y.numpy(), label='sin(x) (truth)', linewidth=2, color='#5cc8ff')
ax2.plot(x.numpy(), pred.numpy(), label='model (learned)', linewidth=2, linestyle='--', color='#6ee7a8')
ax2.legend()
ax2.set_title('Model learned to approximate sin(x)')
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

### Learning rate matters — a lot

Try three learning rates and see what happens.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = {'1e-1 (too high)': '#ef4444', '1e-3 (just right)': '#6ee7a8', '1e-5 (too low)': '#5cc8ff'}

torch.manual_seed(42)
for label, lr in [('1e-1 (too high)', 0.1), ('1e-3 (just right)', 1e-3), ('1e-5 (too low)', 1e-5)]:
    m = nn.Sequential(nn.Linear(1,64), nn.Tanh(), nn.Linear(64,64), nn.Tanh(), nn.Linear(64,1))
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    ls = []
    for _ in range(300):
        opt.zero_grad()
        l = nn.MSELoss()(m(x), y)
        l.backward()
        opt.step()
        ls.append(l.item())
    ax.plot(ls, label=f'lr={label}', color=colors[label], linewidth=2)

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Learning rate comparison')
ax.set_yscale('log')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

---
## 5 · The co-pilot cheat sheet

You now have the vocabulary to work with an AI agent on fine-tuning. Here's what you own vs delegate:

### What YOU decide (the agent needs your input)

| Concept | Why you need it | Example prompt to the agent |
|---------|----------------|----------------------------|
| **Data shape** | Shape mismatches are the #1 error | *"My input is (batch, 5) and target is (batch, 1)"* |
| **Loss behavior** | You watch the curve and diagnose | *"Loss is stuck at 0.5 after 1000 steps, what should I try?"* |
| **Hyperparameters** | You say what to change | *"Try learning rate 2e-4 and rank 16"* |
| **Evaluation** | Low loss ≠ good model | *"Generate 5 test outputs so I can check quality"* |
| **Architecture** | Linear/MLP/Transformer/etc. | *"Use a 2-layer MLP with ReLU for this regression task"* |

### What the AGENT handles (you don't need to memorize)

- Exact API syntax: `model.train()`, `optimizer.zero_grad()`, `loss.backward()`
- Training loop boilerplate
- DataLoader setup and batching
- Default optimizer/scheduler choices
- Library imports and compatibility
- Gradient computation internals

### The mental model

> **You** = the driver (where are we going? are we there yet?)  
> **Agent** = the GPS (turn-by-turn directions, route calculation)  
> **The loss curve** = the windshield (look through it!)

---

**Done?** Open `cases/01_foundations/` to learn about fine-tuning: why a warm start beats a cold start.